# Variational Autoencoder Variants and Extensions

The vanilla VAE is the right starting point, but it is not yet the whole story one wants for a full lecture. Once the ELBO is understood, three deeper questions immediately appear. How strongly should the latent bottleneck be enforced? What happens if we replace the continuous Gaussian code with a discrete representation? And how do we move from **unconditional generation** to **controlled generation**, where the model is told what kind of sample it should produce?

This chapter organizes the VAE landscape around exactly those three questions. Instead of listing many names shallowly, we build three large blocks: **information-control variants**, **discrete-latent variants**, and **conditional variants**. Each block is discussed as a methodological choice, not just as a tweak to a loss function.


This chapter treats each VAE family as a **complete modeling move**. For each one, we first explain what bottleneck or design pressure motivates the variant, then show how the model or the objective changes, and finally include a `FashionMNIST` experiment with qualitative inspection and shared metric evaluation.

The code is intentionally repetitive in places. That is useful here: by keeping the dataset and most of the backbone fixed, the reader can see exactly what changed when moving from `beta-VAE` to annealing, from continuous latents to `VQ-VAE`, or from unconditional to conditional generation.

The training schedules below are also deliberately more serious than “toy notebook” defaults. These variants are not given ten or twenty quick epochs just to produce a figure. The goal is to let them train long enough that the qualitative differences are actually visible.


## Shared Setup

All variants below use the same `FashionMNIST` data pipeline and the same convolutional baseline components, so that comparisons remain meaningful. We also postpone **FID/KID** to the end of the notebook. That way, the real-data features only need to be accumulated once.

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(7)
if device.type == "cuda":
    torch.cuda.manual_seed_all(7)
num_workers = 2 if device.type == "cuda" else 0
project_root = Path.cwd() if (Path.cwd() / "_config.yml").exists() else Path.cwd().parent
DATA_ROOT = project_root / "data"

# Settings
batch_size = 128
latent_dim = 32
base_channels = 32
lr = 2e-4
epochs = 60

transform = transforms.ToTensor()

train_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=True,
    download=True,
    transform=transform,
)

test_dataset = datasets.FashionMNIST(
    root=DATA_ROOT,
    train=False,
    download=True,
    transform=transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=(device.type == "cuda"),
)


In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=32, base_channels=32):
        super().__init__()
        # 28x28 -> 14x14 -> 7x7.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.SiLU(),
            nn.Conv2d(base_channels * 2, base_channels * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.SiLU(),
            nn.Flatten(),
        )
        encoded_dim = base_channels * 4 * 7 * 7
        self.mu_head = nn.Linear(encoded_dim, latent_dim)
        self.logvar_head = nn.Linear(encoded_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, encoded_dim),
            nn.SiLU(),
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.SiLU(),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels),
            nn.SiLU(),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.mu_head(h)
        logvar = self.logvar_head(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        # Sample through parameter-free noise so gradients can flow to mu/logvar.
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        logits = self.decoder(z)
        return logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


model = VAE(latent_dim=latent_dim, base_channels=base_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
@torch.no_grad()
def show_reconstructions(model, loader, device, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    logits, _, _ = model(x)
    # Sigmoid is only for visualization; training used logits directly.
    recon = torch.sigmoid(logits).view(-1, 1, 28, 28)

    grid = torch.cat([x.cpu(), recon.cpu()], dim=0)
    image = utils.make_grid(grid, nrow=n, pad_value=1.0)
    plt.figure(figsize=(1.5 * n, 3.0))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


@torch.no_grad()
def show_samples(model, device, n=16):
    model.eval()
    z = torch.randn(n, latent_dim, device=device)
    logits = model.decode(z)
    samples = torch.sigmoid(logits).view(-1, 1, 28, 28).cpu()
    image = utils.make_grid(samples, nrow=4, pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()

In [ ]:
@torch.no_grad()
def interpolate(model, loader, device, steps=8):
    model.eval()
    x, _ = next(iter(loader))
    x0 = x[0:1].to(device)
    x1 = x[1:2].to(device)

    # Interpolate between posterior means to visualize latent geometry.
    mu0, _ = model.encode(x0)
    mu1, _ = model.encode(x1)

    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * mu0 + alphas * mu1
    logits = model.decode(z)
    images = torch.sigmoid(logits).view(-1, 1, 28, 28).cpu()

    grid = utils.make_grid(images, nrow=steps, pad_value=1.0)
    plt.figure(figsize=(1.7 * steps, 2.5))
    plt.imshow(grid.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


## Controlling Information Flow: Beta-VAE, KL Annealing, and Free Bits

The first major branch of VAE variants does not abandon the basic latent-variable story at all. It asks a more focused question: **how much information should the latent channel be allowed to carry, and when should that information pressure be applied?**

This is a central methodological question because the ELBO is not only a likelihood surrogate. It is also a communication budget between encoder and decoder. If the KL term is too weak, the latent code can become a near-memorization channel. If it is too strong too early, the latent variable may be ignored altogether. The variants in this section should therefore be read as tools for shaping the information flow of the model.


In implementation terms, `beta` is the cleanest control knob in the whole VAE family. It does not change the latent-variable story itself. It changes how expensive latent information becomes. Larger values push the representation toward a more compressed and better regularized latent geometry, but they also increase the risk of losing fine visual detail.

In [ ]:
def kl_per_sample(mu, logvar):
    # Keep the per-sample KL so we can reweight or threshold it flexibly.
    return -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)


def beta_vae_loss(x, logits, mu, logvar, beta=1.0, free_bits=0.0):
    reconstruction = F.binary_cross_entropy_with_logits(
        logits,
        x,
        reduction="sum",
    )
    kl_values = kl_per_sample(mu, logvar)
    if free_bits > 0.0:
        # Each sample is allowed to use a small amount of KL capacity for free.
        kl_values = torch.clamp(kl_values, min=free_bits)
    kl = kl_values.sum()
    loss = reconstruction + beta * kl
    return loss, reconstruction, kl


def linear_beta_schedule(epoch, total_epochs, start=0.0, stop=1.0, warmup_fraction=0.4):
    warmup_epochs = max(1, int(total_epochs * warmup_fraction))
    if epoch >= warmup_epochs:
        return stop
    progress = epoch / warmup_epochs
    return start + progress * (stop - start)


for epoch in range(5):
    beta_t = linear_beta_schedule(epoch, total_epochs=30, start=0.0, stop=4.0)
    print(f"epoch={epoch:02d} beta_t={beta_t:.2f}")

In [ ]:
def train_beta_vae_epoch(model, loader, optimizer, device, beta=1.0):
    model.train()
    total_loss = 0.0
    total_kl = 0.0
    for x, _ in tqdm(loader, desc="beta-VAE train", leave=False):
        x = x.to(device)
        optimizer.zero_grad()
        logits, mu, logvar = model(x)
        loss, _, kl = beta_vae_loss(x, logits, mu, logvar, beta=beta)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_kl += kl.item()
    n = len(loader.dataset)
    return {"loss": total_loss / n, "kl": total_kl / n}


@torch.no_grad()
def evaluate_beta_vae(model, loader, device, beta=1.0):
    model.eval()
    total_loss = 0.0
    total_kl = 0.0
    for x, _ in loader:
        x = x.to(device)
        logits, mu, logvar = model(x)
        loss, _, kl = beta_vae_loss(x, logits, mu, logvar, beta=beta)
        total_loss += loss.item()
        total_kl += kl.item()
    n = len(loader.dataset)
    return {"loss": total_loss / n, "kl": total_kl / n}


beta_model = VAE(latent_dim=latent_dim, base_channels=base_channels).to(device)
beta_optimizer = torch.optim.Adam(beta_model.parameters(), lr=lr)
beta_history = {"train_loss": [], "val_loss": [], "kl": []}
beta_epochs = 50
beta_value = 4.0

for epoch in tqdm(range(beta_epochs), desc="beta-VAE epochs"):
    train_stats = train_beta_vae_epoch(beta_model, train_loader, beta_optimizer, device, beta=beta_value)
    val_stats = evaluate_beta_vae(beta_model, test_loader, device, beta=beta_value)
    beta_history["train_loss"].append(train_stats["loss"])
    beta_history["val_loss"].append(val_stats["loss"])
    beta_history["kl"].append(train_stats["kl"])
    print(
        f"Epoch {epoch + 1:02d} | beta={beta_value:.1f} | "
        f"train loss: {train_stats['loss']:.4f} | "
        f"val loss: {val_stats['loss']:.4f} | "
        f"KL: {train_stats['kl']:.4f}"
    )


A simple but useful experiment is to compare this run qualitatively against the vanilla VAE. One should usually see a more disciplined latent space, often with smoother interpolations, but also some loss of reconstruction fidelity. That trade-off is not a bug in `beta-VAE`; it is the whole point of the variant.

In [ ]:
show_reconstructions(beta_model, test_loader, device)
interpolate(beta_model, test_loader, device)


### KL Annealing, Free Bits, and Posterior Collapse

The next family stays in the same conceptual block, but changes **how the bottleneck is activated during optimization**. A model with a sensible final objective can still fail if the KL pressure is introduced too aggressively from the first iterations. This is the setting in which one starts talking seriously about **posterior collapse**: the decoder learns to model the data while the latent variable becomes almost irrelevant.


In [ ]:
annealed_model = VAE(latent_dim=latent_dim, base_channels=base_channels).to(device)
annealed_optimizer = torch.optim.Adam(annealed_model.parameters(), lr=lr)
annealed_history = {"train_loss": [], "val_loss": [], "beta": []}
annealed_epochs = 50
free_bits_value = 0.1

for epoch in tqdm(range(annealed_epochs), desc="annealed VAE epochs"):
    beta_t = linear_beta_schedule(epoch, annealed_epochs, start=0.0, stop=1.0, warmup_fraction=0.4)
    annealed_model.train()
    total_loss = 0.0
    for x, _ in tqdm(train_loader, desc="annealed train", leave=False):
        x = x.to(device)
        annealed_optimizer.zero_grad()
        logits, mu, logvar = annealed_model(x)
        loss, _, _ = beta_vae_loss(x, logits, mu, logvar, beta=beta_t, free_bits=free_bits_value)
        loss.backward()
        annealed_optimizer.step()
        total_loss += loss.item()
    annealed_history["train_loss"].append(total_loss / len(train_loader.dataset))
    annealed_history["beta"].append(beta_t)
    print(f"Epoch {epoch + 1:02d} | beta_t={beta_t:.2f} | train loss: {annealed_history['train_loss'][-1]:.4f}")


This experiment is conceptually different from the fixed-`beta` one. Here the question is not “what happens if the bottleneck is permanently tighter,” but rather “what happens if the bottleneck is allowed to open first and tighten later.” On real datasets, that scheduling difference can matter a great deal.

`KL` annealing and `free bits` are worth emphasizing because they are not exotic tricks. They are among the most practical ways to keep the latent channel alive long enough for the model to discover useful structure before the prior starts pulling too strongly.


In [ ]:
show_reconstructions(annealed_model, test_loader, device)
interpolate(annealed_model, test_loader, device)


The scheduling function is especially important to discuss slowly. A fixed `beta=4` from the first iteration and a run that *ends* near `beta=4` after a warm-up are not the same thing. In the first case, the optimizer may shut down the latent channel before it ever becomes useful. In the second case, the model first learns to communicate and only later tightens the bottleneck.

Taken together, `beta-VAE`, `KL` annealing, and `free bits` teach a common lesson: VAE design is not only about the existence of a latent variable, but about **how much information is allowed to pass through it and on what schedule**.


## VQ-VAE and Discrete Latent Spaces

The second major branch of VAE variants changes something more radical than the information-control family. A `VQ-VAE` does not merely reweight the ELBO. It changes the **type of latent representation** itself, moving from a continuous Gaussian code to a learned discrete codebook.

That shift matters because it changes both the geometry of the latent space and the way the model can later be combined with other generative components. This is one reason `VQ-VAE` became such an important bridge toward modern latent generative pipelines.


### Why Discrete Latents Matter

`VQ-VAE` deserves a fuller description because it changes the *kind* of latent variable rather than only reweighting an existing objective. Instead of sampling from a Gaussian latent posterior, the encoder output is quantized to the nearest entry of a learned codebook. The decoder therefore receives a discrete symbolic representation. This changes the geometry of the latent space substantially: nearby encoder outputs may snap to the same code, and the latent representation begins to look more like a learned vocabulary than like a smooth Euclidean cloud.

```{figure} ../assets/images/approximation.png
:width: 62%
:align: center

A schematic reminder that generative modeling is often about finding a tractable approximation to a complicated target distribution. VQ-VAE changes that approximation story by replacing continuous Gaussian latents with a learned discrete codebook.
```


A proper `VQ-VAE` implementation introduces three extra ingredients beyond the vanilla VAE story: a codebook, a nearest-neighbor quantization step, and codebook/commitment losses. But once that tokenizer-like latent representation exists, there is a second stage that matters just as much: learning a **prior over code sequences**. Without that prior, one can reconstruct images and inspect the codebook, but unconditional generation remains weak.

So below we do the complete thing: first train the `VQ-VAE` itself, then freeze it, encode the training set into discrete index grids, and finally train a small autoregressive prior over those grids. That is the minimum setup that actually demonstrates why discrete latents are useful rather than just interesting.


In [ ]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=256, embedding_dim=64, commitment_cost=0.25):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.commitment_cost = commitment_cost
        self.codebook = nn.Embedding(num_embeddings, embedding_dim)
        self.codebook.weight.data.uniform_(-1 / num_embeddings, 1 / num_embeddings)

    def forward(self, z_e):
        # Flatten spatial locations so each one chooses its nearest codebook vector.
        flat = z_e.permute(0, 2, 3, 1).contiguous().view(-1, self.embedding_dim)
        distances = (
            flat.pow(2).sum(1, keepdim=True)
            - 2 * flat @ self.codebook.weight.t()
            + self.codebook.weight.pow(2).sum(1)
        )
        indices = distances.argmin(dim=1)
        z_q = self.codebook(indices).view(z_e.size(0), z_e.size(2), z_e.size(3), self.embedding_dim)
        z_q = z_q.permute(0, 3, 1, 2).contiguous()

        commitment = F.mse_loss(z_e.detach(), z_q)
        codebook = F.mse_loss(z_e, z_q.detach())
        loss = codebook + self.commitment_cost * commitment

        # Straight-through estimator: decoder sees quantized codes, encoder keeps gradients.
        z_q = z_e + (z_q - z_e).detach()
        return z_q, loss, indices.view(z_e.size(0), z_e.size(2), z_e.size(3))


class VQVAE(nn.Module):
    def __init__(self, embedding_dim=64, num_embeddings=256, base_channels=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(base_channels, embedding_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
        )
        self.quantizer = VectorQuantizer(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels, 1, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices = self.quantizer(z_e)
        logits = self.decoder(z_q)
        return logits, vq_loss, indices


class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.register_buffer("mask", torch.ones_like(self.weight))
        _, _, h, w = self.weight.shape
        self.mask[:, :, h // 2, w // 2 + (mask_type == "B") :] = 0
        self.mask[:, :, h // 2 + 1 :] = 0

    def forward(self, x):
        masked_weight = self.weight * self.mask
        return F.conv2d(
            x,
            masked_weight,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )


class PixelCNNPrior(nn.Module):
    def __init__(self, num_embeddings, hidden_channels=128):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, hidden_channels)
        self.net = nn.Sequential(
            MaskedConv2d("A", hidden_channels, hidden_channels, kernel_size=5, padding=2),
            nn.ReLU(True),
            MaskedConv2d("B", hidden_channels, hidden_channels, kernel_size=5, padding=2),
            nn.ReLU(True),
            MaskedConv2d("B", hidden_channels, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU(True),
            nn.Conv2d(hidden_channels, num_embeddings, kernel_size=1),
        )

    def forward(self, indices):
        x = self.embedding(indices).permute(0, 3, 1, 2).contiguous()
        return self.net(x)


@torch.no_grad()
def encode_dataset_to_indices(model, loader, device):
    model.eval()
    all_indices = []
    for x, _ in tqdm(loader, desc="encode codes", leave=False):
        x = x.to(device)
        _, _, indices = model(x)
        all_indices.append(indices.cpu())
    return torch.cat(all_indices, dim=0)


vqvae = VQVAE().to(device)
vq_optimizer = torch.optim.Adam(vqvae.parameters(), lr=2e-4)
vq_epochs = 60
for epoch in tqdm(range(vq_epochs), desc="VQ-VAE epochs"):
    vqvae.train()
    running = 0.0
    for x, _ in tqdm(train_loader, desc="VQ train", leave=False):
        x = x.to(device)
        vq_optimizer.zero_grad()
        logits, vq_loss, _ = vqvae(x)
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="mean")
        loss = recon + vq_loss
        loss.backward()
        vq_optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch + 1:02d} | VQ-VAE loss: {running / len(train_loader):.4f}")


train_code_indices = encode_dataset_to_indices(vqvae, train_loader, device)
test_code_indices = encode_dataset_to_indices(vqvae, test_loader, device)
code_height, code_width = train_code_indices.shape[1:]

prior_train_loader = DataLoader(
    TensorDataset(train_code_indices.long()),
    batch_size=256,
    shuffle=True,
    num_workers=0,
)
prior_test_loader = DataLoader(
    TensorDataset(test_code_indices.long()),
    batch_size=256,
    shuffle=False,
    num_workers=0,
)

pixel_prior = PixelCNNPrior(vqvae.quantizer.num_embeddings).to(device)
prior_optimizer = torch.optim.Adam(pixel_prior.parameters(), lr=2e-4)
prior_epochs = 35
for epoch in tqdm(range(prior_epochs), desc="PixelCNN prior epochs"):
    pixel_prior.train()
    running = 0.0
    for (indices,) in tqdm(prior_train_loader, desc="prior train", leave=False):
        indices = indices.to(device)
        prior_optimizer.zero_grad()
        logits = pixel_prior(indices)
        loss = F.cross_entropy(logits, indices)
        loss.backward()
        prior_optimizer.step()
        running += loss.item()
    prior_eval = 0.0
    pixel_prior.eval()
    for (indices,) in prior_test_loader:
        indices = indices.to(device)
        logits = pixel_prior(indices)
        prior_eval += F.cross_entropy(logits, indices).item()
    prior_eval /= len(prior_test_loader)
    print(f"Epoch {epoch + 1:02d} | prior nll: {running / len(prior_train_loader):.4f} | val nll: {prior_eval:.4f}")


@torch.no_grad()
def sample_vqvae_with_prior(model, prior, n=16, return_cpu=False, visualize=False):
    model.eval()
    prior.eval()
    indices = torch.zeros(n, code_height, code_width, dtype=torch.long, device=device)
    for row in range(code_height):
        for col in range(code_width):
            logits = prior(indices)
            probs = torch.softmax(logits[:, :, row, col], dim=1)
            indices[:, row, col] = torch.multinomial(probs, num_samples=1).squeeze(1)
    z_q = model.quantizer.codebook(indices.view(-1)).view(
        n, code_height, code_width, model.quantizer.embedding_dim
    )
    z_q = z_q.permute(0, 3, 1, 2).contiguous()
    logits = model.decoder(z_q)
    samples = torch.sigmoid(logits)
    if visualize:
        image = utils.make_grid(samples.cpu(), nrow=4, pad_value=1.0)
        plt.figure(figsize=(6, 6))
        plt.imshow(image.permute(1, 2, 0), cmap="gray")
        plt.axis("off")
        plt.show()
    return samples.cpu() if return_cpu else samples


In [ ]:
class VQVAE(nn.Module):
    def __init__(self, embedding_dim=64, num_embeddings=256, base_channels=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.Conv2d(base_channels, embedding_dim, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
        )
        self.quantizer = VectorQuantizer(num_embeddings=num_embeddings, embedding_dim=embedding_dim)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embedding_dim, base_channels, kernel_size=4, stride=2, padding=1),
            nn.ReLU(True),
            nn.ConvTranspose2d(base_channels, 1, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices = self.quantizer(z_e)
        logits = self.decoder(z_q)
        return logits, vq_loss, indices


@torch.no_grad()
def collect_code_histogram(model, loader, device, num_embeddings=256):
    counts = torch.zeros(num_embeddings, device=device)
    model.eval()
    for x, _ in loader:
        x = x.to(device)
        _, _, indices = model(x)
        counts += torch.bincount(indices.view(-1), minlength=num_embeddings).float()
    probs = counts / counts.sum()
    return probs


vqvae = VQVAE().to(device)
vq_optimizer = torch.optim.Adam(vqvae.parameters(), lr=2e-4)
vq_epochs = 20
for epoch in tqdm(range(vq_epochs), desc="VQ-VAE epochs"):
    vqvae.train()
    running = 0.0
    for x, _ in tqdm(train_loader, desc="VQ train", leave=False):
        x = x.to(device)
        vq_optimizer.zero_grad()
        logits, vq_loss, _ = vqvae(x)
        recon = F.binary_cross_entropy_with_logits(logits, x, reduction="mean")
        loss = recon + vq_loss
        loss.backward()
        vq_optimizer.step()
        running += loss.item()
    print(f"Epoch {epoch + 1:02d} | VQ-VAE loss: {running / len(train_loader):.4f}")

code_probs = collect_code_histogram(vqvae, train_loader, device)


@torch.no_grad()
def sample_vqvae(model, code_probs, n=16, h=7, w=7):
    model.eval()
    indices = torch.multinomial(code_probs, n * h * w, replacement=True)
    z_q = model.quantizer.codebook(indices).view(n, h, w, model.quantizer.embedding_dim)
    z_q = z_q.permute(0, 3, 1, 2).contiguous()
    logits = model.decoder(z_q)
    samples = torch.sigmoid(logits).cpu()
    image = utils.make_grid(samples, nrow=4, pad_value=1.0)
    plt.figure(figsize=(6, 6))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


Once the prior is trained, `VQ-VAE` becomes a real two-stage generative pipeline rather than just an autoencoding tokenizer. The first stage learns a discrete latent vocabulary. The second stage learns which spatial arrangements of those codes are actually probable under the data distribution.

That distinction matters in practice. Reconstruction quality mostly tests the encoder-decoder pair. Unconditional generation quality depends heavily on the prior. If the prior is weak, the codebook may be good while sampling is still poor. If the prior is strong, the discrete latent space becomes a compact interface on top of which sampling can be done much more effectively.


In [ ]:
@torch.no_grad()
def show_vqvae_reconstructions(model, loader, device, n=8):
    model.eval()
    x, _ = next(iter(loader))
    x = x[:n].to(device)
    logits, _, _ = model(x)
    recon = torch.sigmoid(logits).cpu()
    grid = utils.make_grid(torch.cat([x.cpu(), recon], dim=0), nrow=n, pad_value=1.0)
    plt.figure(figsize=(1.5 * n, 3.0))
    plt.imshow(grid.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()


show_vqvae_reconstructions(vqvae, test_loader, device)
sample_vqvae_with_prior(vqvae, pixel_prior, n=16, visualize=True)


## Conditional VAEs and the Move to Controlled Generation

We postpone the conditional VAE to the end on purpose, because it is not just another unconditional variant. It changes the task itself. Instead of modeling a single distribution $p_\theta(\boldsymbol{x})$, we now want to model a family of distributions $p_\theta(\boldsymbol{x} \mid \boldsymbol{y})$, where $\boldsymbol{y}$ is some observed condition such as a class label, a text description, or another modality.

Methodologically, this is one of the most important steps in modern generative modeling. Unconditional generation asks the model to produce *some* plausible sample. Conditional generation asks the model to produce a plausible sample that is also **compatible with a user-specified constraint**. That is the conceptual bridge toward prompted generation in later systems: the prompt does not remove stochasticity, it reshapes the target distribution.


In [ ]:
num_classes = 10


class ConditionalVAE(nn.Module):
    def __init__(self, latent_dim=32, base_channels=32, num_classes=10, label_dim=16):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, label_dim)
        self.encoder = nn.Sequential(
            nn.Conv2d(1, base_channels, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
            nn.Flatten(),
        )
        encoded_dim = base_channels * 2 * 7 * 7
        self.mu_head = nn.Linear(encoded_dim + label_dim, latent_dim)
        self.logvar_head = nn.Linear(encoded_dim + label_dim, latent_dim)
        self.decoder_input = nn.Linear(latent_dim + label_dim, base_channels * 4 * 7 * 7)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (base_channels * 4, 7, 7)),
            nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
            nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_channels, 1, kernel_size=3, padding=1),
        )

    def encode(self, x, y):
        h = self.encoder(x)
        y_embed = self.label_embed(y)
        h = torch.cat([h, y_embed], dim=1)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z, y):
        y_embed = self.label_embed(y)
        h = self.decoder_input(torch.cat([z, y_embed], dim=1))
        return self.decoder(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z, y)
        return logits, mu, logvar


@torch.no_grad()
def sample_cvae(model, class_id, n, device):
    model.eval()
    labels = torch.full((n,), class_id, device=device, dtype=torch.long)
    z = torch.randn(n, latent_dim, device=device)
    logits = model.decode(z, labels)
    return torch.sigmoid(logits)


@torch.no_grad()
def sample_cvae_from_labels(model, labels, device):
    model.eval()
    z = torch.randn(labels.size(0), latent_dim, device=device)
    logits = model.decode(z, labels)
    return torch.sigmoid(logits)


In [ ]:
def train_conditional_vae_epoch(model, loader, optimizer, device, beta=1.0, free_bits=0.0):
    model.train()
    total_loss = 0.0
    total_kl = 0.0

    for x, y in tqdm(loader, desc="cVAE train", leave=False):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits, mu, logvar = model(x, y)
        loss, _, kl = beta_vae_loss(x, logits, mu, logvar, beta=beta, free_bits=free_bits)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_kl += kl.item()

    n = len(loader.dataset)
    return {"loss": total_loss / n, "kl": total_kl / n}


@torch.no_grad()
def evaluate_conditional_vae(model, loader, device, beta=1.0, free_bits=0.0):
    model.eval()
    total_loss = 0.0
    total_kl = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, mu, logvar = model(x, y)
        loss, _, kl = beta_vae_loss(x, logits, mu, logvar, beta=beta, free_bits=free_bits)
        total_loss += loss.item()
        total_kl += kl.item()

    n = len(loader.dataset)
    return {"loss": total_loss / n, "kl": total_kl / n}


conditional_epochs = 50
cvae = ConditionalVAE(
    latent_dim=latent_dim,
    base_channels=base_channels,
    num_classes=num_classes,
).to(device)
cvae_optimizer = torch.optim.Adam(cvae.parameters(), lr=lr)
conditional_history = {"train_loss": [], "val_loss": [], "beta": []}

for epoch in tqdm(range(conditional_epochs), desc="cVAE epochs"):
    beta_t = linear_beta_schedule(epoch, conditional_epochs, start=0.0, stop=1.0, warmup_fraction=0.4)
    train_stats = train_conditional_vae_epoch(cvae, train_loader, cvae_optimizer, device, beta=beta_t, free_bits=0.1)
    val_stats = evaluate_conditional_vae(cvae, test_loader, device, beta=beta_t, free_bits=0.1)

    conditional_history["train_loss"].append(train_stats["loss"])
    conditional_history["val_loss"].append(val_stats["loss"])
    conditional_history["beta"].append(beta_t)

    print(
        f"Epoch {epoch + 1:02d} | beta: {beta_t:.2f} | "
        f"train loss: {train_stats['loss']:.4f} | val loss: {val_stats['loss']:.4f}"
    )


At this point, the label is not a decorative extra input. It changes the generative question from “draw any fashion item” to “draw a fashion item of the requested class.” The latent variable still matters, but its role changes: it no longer explains *which class* the object belongs to, because that part is already specified by the condition. Instead, it models the residual uncertainty *inside* the class.

This is why conditioning is so important conceptually. Once a condition is given, generation becomes a controlled mapping from a coarse semantic request plus latent noise to a concrete sample. In tiny form, that is already the same idea that later appears in text-guided image generation: the condition determines the semantic target, while the latent randomness governs the particular realization.


In [ ]:
@torch.no_grad()
def show_cvae_class_samples(model, class_names, device, n_per_class=6):
    model.eval()
    rows = []
    for class_id, _ in enumerate(class_names):
        labels = torch.full((n_per_class,), class_id, device=device, dtype=torch.long)
        z = torch.randn(n_per_class, latent_dim, device=device)
        logits = model.decode(z, labels)
        rows.append(torch.sigmoid(logits).cpu())

    image = utils.make_grid(torch.cat(rows, dim=0), nrow=n_per_class, pad_value=1.0)
    plt.figure(figsize=(1.6 * n_per_class, 1.6 * len(class_names)))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()

    for class_id, name in enumerate(class_names):
        print(f"row {class_id}: {name}")


@torch.no_grad()
def interpolate_cvae_with_fixed_class(model, class_id, class_name, device, steps=8):
    model.eval()
    labels = torch.full((2,), class_id, device=device, dtype=torch.long)
    z0 = torch.randn(1, latent_dim, device=device)
    z1 = torch.randn(1, latent_dim, device=device)
    alphas = torch.linspace(0, 1, steps, device=device).view(-1, 1)
    z = (1 - alphas) * z0 + alphas * z1
    labels = torch.full((steps,), class_id, device=device, dtype=torch.long)
    logits = model.decode(z, labels)
    samples = torch.sigmoid(logits).cpu()
    image = utils.make_grid(samples, nrow=steps, pad_value=1.0)
    plt.figure(figsize=(1.7 * steps, 2.5))
    plt.imshow(image.permute(1, 2, 0), cmap="gray")
    plt.axis("off")
    plt.show()
    print(f"class-conditioned interpolation: {class_name}")


show_cvae_class_samples(cvae, train_dataset.classes, device, n_per_class=6)
interpolate_cvae_with_fixed_class(cvae, class_id=9, class_name=train_dataset.classes[9], device=device)


The expected qualitative result is now stronger than in the unconditional case. Each row should stay semantically tied to its class, while still showing variability across columns. For instance, the `Sandal` row should not collapse to a single repeated template, and it should not drift into `Sneaker` or `Ankle boot`. If that happens, the model is either undertrained or not using the conditioning signal effectively.

The interpolation experiment adds a second check. By fixing the class and moving only in latent space, we can ask whether the model learned a smooth **within-class manifold**. A good result should morph one boot into another boot, not silently wander into a different class. That is exactly the decomposition we want: the label controls the *semantic family*, and the latent variable controls the *instance-level variation* inside that family.


This methodological separation between **condition information** and **residual uncertainty** is the real reason the conditional VAE matters. It is not only a convenient classroom example. It is the point where generative modeling becomes interactively steerable. Once that idea is understood, later conditioning mechanisms in diffusion models or multimodal generators become much easier to interpret.


## Shared FID/KID Evaluation With Cached Real Features

All the qualitative experiments above can be run variant by variant. For **FID** and **KID**, however, it is more efficient to centralize the evaluation here and cache the real-data features once. That is exactly the right strategy for a notebook that compares several generators on the same dataset.

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance


def prepare_for_inception_metrics(images):
    if images.size(1) == 1:
        images = images.repeat(1, 3, 1, 1)
    return images.clamp(0.0, 1.0)


@torch.no_grad()
def build_cached_real_metrics(real_loader, device, num_real=1000, metric_batch_size=64):
    fid = FrechetInceptionDistance(feature=2048, normalize=True, reset_real_features=False).set_dtype(torch.float64).to(device)
    kid = KernelInceptionDistance(feature=2048, subsets=10, subset_size=100, normalize=True, reset_real_features=False).to(device)
    seen = 0
    for real_images, _ in tqdm(real_loader, desc="real metric cache", leave=False):
        remaining = num_real - seen
        if remaining <= 0:
            break
        real_images = real_images[: min(metric_batch_size, remaining)].to(device)
        real_images = prepare_for_inception_metrics(real_images)
        fid.update(real_images, real=True)
        kid.update(real_images, real=True)
        seen += real_images.size(0)
    return fid, kid


@torch.no_grad()
def evaluate_with_cached_real(sample_fn, base_fid, base_kid, num_fake=1000, metric_batch_size=64):
    fid = copy.deepcopy(base_fid)
    kid = copy.deepcopy(base_kid)
    generated = 0
    pbar = tqdm(total=num_fake, desc="fake metric pass", leave=False)
    while generated < num_fake:
        batch_n = min(metric_batch_size, num_fake - generated)
        fake_images = prepare_for_inception_metrics(sample_fn(batch_n).to(device))
        fid.update(fake_images, real=False)
        kid.update(fake_images, real=False)
        generated += batch_n
        pbar.update(batch_n)
    pbar.close()
    kid_mean, kid_std = kid.compute()
    return {"fid": fid.compute().item(), "kid_mean": kid_mean.item(), "kid_std": kid_std.item()}


base_fid, base_kid = build_cached_real_metrics(test_loader, device)

beta_scores = evaluate_with_cached_real(
    lambda n: torch.sigmoid(beta_model.decode(torch.randn(n, latent_dim, device=device))).view(-1, 1, 28, 28),
    base_fid, base_kid,
)
annealed_scores = evaluate_with_cached_real(
    lambda n: torch.sigmoid(annealed_model.decode(torch.randn(n, latent_dim, device=device))).view(-1, 1, 28, 28),
    base_fid, base_kid,
)
cvae_scores = evaluate_with_cached_real(
    lambda n: sample_cvae_from_labels(cvae, torch.randint(0, num_classes, (n,), device=device), device=device),
    base_fid, base_kid,
)
vq_scores = evaluate_with_cached_real(
    lambda n: sample_vqvae_with_prior(vqvae, pixel_prior, n=n),
    base_fid, base_kid,
)

print({"beta_vae": beta_scores, "annealed_vae": annealed_scores, "cvae": cvae_scores, "vqvae": vq_scores})


## What To Remember

For a first substantial VAE module, three variant families are enough to organize the landscape well.

1. **Information-control variants** such as `beta-VAE`, `KL` annealing, and `free bits` teach how strongly the model should compress information and how that pressure should be scheduled during training.
2. **Discrete-latent variants** such as `VQ-VAE` teach that one can redesign the latent space itself, not only reweight the ELBO.
3. **Conditional variants** such as the `CVAE` teach how generative modeling becomes controlled rather than unconditional, which is a key conceptual precursor to modern prompted generation.

That is a better teaching path than listing many names shallowly. These three blocks are common, conceptually distinct, and rich enough to support both theory and implementation.
